Week 3 made lexical search measurable. This week changes both the unit being retrieved and the signals used to rank it. Those are separate interventions: chunking changes what evidence can fit into context, while embeddings change which candidates can be found. Separate comparisons make it possible to attribute a gain to chunking, semantic retrieval, or their combination.

The reference is the [Week 4 release](https://github.com/jamwithai/production-agentic-rag-course/tree/week4.0): `src/services/indexing/`, `src/services/embeddings/`, `src/services/opensearch/index_config_hybrid.py`, and `src/routers/hybrid_search.py`. The objective is to explain a chunk boundary, verify an embedding contract, and compare fused rankings against the lexical baseline.

## A chunk is a retrieval and evidence unit

A full paper often exceeds the context space available for a question. Retrieving the whole paper also mixes the useful paragraph with background, references, and unrelated experiments. A chunk is a smaller passage that can be independently ranked and attributed.

Smaller chunks improve focus but can remove necessary context. “This method improves recall by 12%” is hard to interpret without the method, baseline, dataset, and table heading. Larger chunks preserve more relationships but increase retrieval noise and context consumption. The correct boundary is therefore partly semantic: sections, paragraphs, tables, and definitions are useful units before word counts are applied.

The source chunker first tries section-based splitting and falls back to word-based windows. Its defaults include a 600-word target and 100-word overlap. These are implementation choices, not universal optima. Words also are not model tokens. A token budget must eventually be measured using the generation model's tokenizer, especially for equations, code, and identifiers.

**Overlap** places some text in adjacent chunks to reduce boundary loss. With window size $w$ and overlap $o$, the stride is $w-o$, which must be positive. For a long sequence, the approximate storage multiplier is $w/(w-o)$, ignoring the end. At 600 and 100, that is about 1.2. Overlap also produces near-duplicate retrieval hits, which can crowd independent evidence out of a top-$k$ context.

**Experiment: preserve a fact crossing a boundary.** The integer positions stand in for tokens. The first split separates a two-position fact; overlap makes it available together.

1. Run the cell and compare the two chunkings.
2. Change the fact so it is longer than the overlap.
3. Check which chunkings still preserve it.


In [ ]:
def windows(sequence, width, overlap):
    if width <= 0 or not 0 <= overlap < width:
        raise ValueError("Require width > 0 and 0 <= overlap < width")
    result = []
    for start in range(0, len(sequence), width - overlap):
        chunk = sequence[start:start + width]
        result.append(chunk)
        if start + width >= len(sequence):
            break
    return result

sequence = list(range(18))
fact = {5, 6}
for overlap in (0, 2):
    chunks = windows(sequence, width=6, overlap=overlap)
    preserved = any(fact <= set(chunk) for chunk in chunks)
    print("overlap", overlap, "chunks", chunks, "fact preserved", preserved)
assert not any(fact <= set(c) for c in windows(sequence, 6, 0))
assert any(fact <= set(c) for c in windows(sequence, 6, 2))


Reference output from isolated Python execution:

```text
overlap 0 chunks [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17]] fact preserved False
overlap 2 chunks [[0, 1, 2, 3, 4, 5], [4, 5, 6, 7, 8, 9], [8, 9, 10, 11, 12, 13], [12, 13, 14, 15, 16, 17]] fact preserved True
```

This establishes a boundary mechanism, not improved retrieval quality. A real parser may lose the table heading before chunking begins, and overlap cannot restore missing text. Parent paper ID, source version, section, and offsets make a result inspectable in its original context. Approximate offsets produced after whitespace normalization should not be presented as exact source coordinates.

An indexing revision needs a policy for old chunks. Stable IDs such as `(source_version, chunker_version, chunk_index)` help identify artifacts, but replacement also must remove obsolete chunks when a new version produces fewer of them. Otherwise the index can mix incompatible representations of the same paper.

## Embeddings have an interface, not just a dimension

An embedding maps input text to a vector. Similarity between vectors is useful only when the representations were produced under compatible assumptions: model, version, dimensions, normalization, input task, and preprocessing. Two models producing 1,024 numbers do not necessarily share a coordinate system.

The source uses Jina embeddings with separate document and query tasks. The distinction between `embed_query` and passage embedding reflects the different roles of a question and a document; identical API settings are not automatically appropriate for both. The OpenSearch mapping expects 1,024-dimensional vectors and cosine similarity. Changing that dimension or model generally requires a new compatible index and re-embedding; changing only the query side creates an invalid comparison.

The Week 4 notebook contains a fallback that returns the same constant vector when an API key is absent or a request fails. It preserves array shape, but it contains no semantic information. All texts receiving the same nonzero vector have cosine similarity one. A successful request through that path tests plumbing, not semantic retrieval.

**Experiment: expose the placeholder failure.** The vectors here are deliberately hand-designed. They illustrate geometry only and must never be described as learned semantic embeddings.


In [ ]:
import math

def cosine(left, right):
    if len(left) != len(right):
        raise ValueError("Dimension mismatch")
    left_norm = math.sqrt(sum(x*x for x in left))
    right_norm = math.sqrt(sum(x*x for x in right))
    if not left_norm or not right_norm:
        raise ValueError("Zero vectors have undefined cosine similarity")
    return sum(a*b for a, b in zip(left, right)) / (left_norm * right_norm)

constant = [0.1] * 8
assert math.isclose(cosine(constant, constant), 1.0)
query = [1.0, 0.0]
print("Constant fallback similarity:", round(cosine(constant, constant), 6))
print("Designed nearby vector:", round(cosine(query, [0.9, 0.1]), 6))
print("Designed unrelated vector:", cosine(query, [0.0, 1.0]))


Reference output from isolated Python execution:

```text
Constant fallback similarity: 1.0
Designed nearby vector: 0.993884
Designed unrelated vector: 0.0
```

Correct shape is necessary but insufficient. Checks for finite values, correct dimensions, nonzero norms, and different texts producing meaningfully different vectors can expose violations of the embedding contract. Those checks catch obvious defects, but semantic relevance still requires a labeled retrieval experiment. Fallback telemetry distinguishes successful semantic retrieval from a shape-preserving placeholder, which cannot demonstrate an improvement over BM25.

## Rank fusion across different score scales

BM25 scores and cosine similarities have different scales and meanings. Adding them with arbitrary weights can cause the numerically larger scale to dominate. Even min-max normalization depends on the candidate set and can be unstable across queries.

**Reciprocal rank fusion** combines positions rather than raw scores:

$$
\operatorname{RRF}(d)=\sum_{j:\,d\in L_j}\frac{1}{c+r_j(d)},
$$

where $L_j$ is a ranked list, $r_j(d)$ is the one-based position of document $d$ in that list, and $c>0$ softens the influence of very early positions. An item absent from a list contributes zero. A document ranked reasonably well by both retrievers can outrank one ranked first by only one.

The source defines an OpenSearch `score-ranker-processor` with RRF and rank constant 60. OpenSearch introduced that processor in [version 2.19](https://docs.opensearch.org/2.19/search-plugins/search-pipelines/score-ranker-processor/). The notebook's printed description of “60% BM25 + 40% Vector” does not describe that rank-fusion configuration. The number 60 is a denominator constant, not a percentage.

**Experiment: calculate agreement explicitly.** Each input list contains unique chunk IDs, so each retriever contributes at most once per item.

1. Calculate the contributions for B.
2. Compare their sum with the first-place contribution for A.
3. Run the cell to check the predicted ordering.


In [ ]:
from collections import defaultdict

ranked_lists = {"lexical": ["A", "B", "C"], "vector": ["D", "B", "E"]}
fused = defaultdict(float)
for ranked in ranked_lists.values():
    assert len(ranked) == len(set(ranked))
    for rank, item in enumerate(ranked, 1):
        fused[item] += 1 / (60 + rank)
ordering = sorted(fused, key=lambda item: (-fused[item], item))
for item in ordering:
    print(item, f"{fused[item]:.6f}")
assert ordering[0] == "B"
assert math.isclose(fused["B"], 2 / 62)


Reference output from isolated Python execution:

```text
B 0.032258
A 0.016393
D 0.016393
C 0.015873
E 0.015873
```

Agreement wins in this example because B appears near the top of both lists. This does not mean RRF always improves quality. It cannot recover evidence missing from every candidate list. Candidate depth, approximate nearest-neighbor recall, filters, and duplicate chunks all shape the lists before fusion begins.

OpenSearch uses an approximate vector index in this release. Approximation trades indexing resources and query cost against recall. A perfect fusion implementation can still fail when the vector search never returns the relevant neighbor. Candidate retrieval and final ranking consequently need distinct measurements.

## Filters must constrain every retrieval path

The `_search_hybrid_native` implementation in [`client.py`](https://github.com/jamwithai/production-agentic-rag-course/blob/week4.0/src/services/opensearch/client.py) makes the filter boundary visible. The lexical clause is built with category filters, but the adjacent vector `knn` clause does not visibly carry the same category constraint. That raises a concrete correctness question: can a vector-only candidate outside the requested category enter the fused result?

This matters beyond categories. If the filter encoded tenant or document permissions, applying it to only one branch could reveal ineligible content. Rank fusion is not an authorization mechanism. Eligibility must be consistently enforced before evidence reaches the generator, with a final defense-in-depth check where appropriate.

The response's `total` field presents another contract question. The hybrid branch overwrites it with the number of retained hits, while other search paths may report an index hit count. A shared field name does not guarantee shared semantics. A useful API states whether a number is the returned count, an estimated match count, or a count after postprocessing.

## Docker lab: separate chunking, vectors, and fusion

1. Prepare the matching `week4.0` checkout, project name `rag-week4`, and a real Jina API key in `.env`. Do not commit or print the key. Stop previous stacks, start the complete Compose definition, and ingest a small corpus through the Airflow DAG. Verify that a chunk index, rather than only the earlier paper index, is populated.

    ```bash
    docker compose --project-name rag-week4 up --build -d
    docker compose --project-name rag-week4 exec airflow \
      airflow dags trigger arxiv_paper_ingestion
    curl --fail-with-body http://localhost:9200/arxiv-papers-chunks/_mapping
    curl --fail-with-body http://localhost:9200/arxiv-papers-chunks/_count
    curl --fail-with-body http://localhost:9200/_search/pipeline/hybrid-rrf-pipeline
    curl --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:8000/api/v1/hybrid-search/ \
      -d '{"query":"retrieval augmented language models","size":5,"use_hybrid":false}'
    curl --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:8000/api/v1/hybrid-search/ \
      -d '{"query":"retrieval augmented language models","size":5,"use_hybrid":true}'
    ```

2. Inspect a few stored chunks and vectors. Confirm dimensions and model metadata; compare chunks from different papers. The API key must actually be used successfully. A request labeled hybrid after an embedding failure is not a valid hybrid measurement. Trace the branch taken and the search-pipeline parameter.

3. Run three comparisons on the same fixed questions and corpus: paper-level versus chunk-level lexical retrieval; chunk-level lexical versus vector retrieval; and their fused ranking. Report passage evidence coverage and paper diversity as well as latency. This isolates the cause of any gain instead of changing chunk size, model, and ranker simultaneously.

4. Create two controlled test records in a disposable test index with different categories and distinct vectors. Submit a category-limited hybrid query whose nearest vector belongs to the excluded category. The expected invariant is zero excluded-category results. If it fails, preserve the request and mapping as a minimal reproducer before modifying query construction.

5. Stop or misconfigure the embedding dependency temporarily and repeat the query. Check whether the system explicitly falls back to BM25, fails clearly, or silently loses results. Restore the configuration afterward. Each is observable behavior; only the first two make the degraded mode clear enough to evaluate.

## Engineering review: justify the extra complexity

1. Choose a query that lexical search misses and explain exactly which passage a vector retriever should recover. Verify that the passage exists after parsing and chunking before judging the embedding model. Then choose an exact identifier that should favor lexical retrieval. Hybrid search earns its place if the combined system improves the required query categories at an acceptable cost.

2. Review short-text behavior in `TextChunker.chunk_text`: the small-input branch calls `_reconstruct_text` with a different argument count from its definition in this source. This is a useful boundary test to write before trusting the larger pipeline. Reproducing and repairing that case is an engineering exercise; the lesson does not modify the upstream implementation.

3. Specify a migration plan for changing embedding models: a separate compatible index, complete backfill, quality checks, a controlled switch, and a rollback path. Include handling for deletions and obsolete chunks. The ability to repeat and reverse a representation change is more valuable than treating today's default dimension as permanent.

## Sources and scope

- [Week 4 source](https://github.com/jamwithai/production-agentic-rag-course/tree/week4.0): chunking, embeddings, hybrid queries, mapping, and pipeline configuration.
- [OpenSearch 2.19 score ranker](https://docs.opensearch.org/2.19/search-plugins/search-pipelines/score-ranker-processor/): RRF parameters and version requirements.
- [Jina API reference](https://docs.jina.ai/): confirm the live model and task contract when conducting the integration lab; the course source remains the reference for the historical configuration.

The vector and fusion cells verify arithmetic and counterexamples. They do not simulate a learned embedding model or establish approximate-search recall. Those claims require the Docker system, actual embeddings, and relevance judgments.
